<div align="center">

# WEEK 03
## A3 DATA INTEGRATION & DATASET MERGING: STAR SCHEMA

### CadetX Virtual Work Experience
### Heavy Supplier, Inventory & Warehouse Analytics

</div>

---

| | |
|---|---|
| **Prepared by** | Keren Oyem |
| **Sprint** | Week 3 · Fri 6 Nov – Thu 12 Nov 2026 |
| **Catalogue task** | A3 Data Integration & Dataset Merging |
| **Input** | `data/processed/*.parquet` (Week 2 clean layer) |
| **Output** | Schema `star` inside `warehouse.duckdb` (5 dimensions + 4 facts) |
| **Tools** | Python, SQL on DuckDB |

## Purpose

Week 2 produced 12 clean tables. This notebook joins them into an **analytical star schema** so that Weeks 4–12 (KPIs, forecasting, dashboards) can query one tidy model instead of 12 separate tables.

* **Dimensions** describe *who / what / where / when*: `dim_date`, `dim_product`, `dim_branch`, `dim_customer`, `dim_supplier`.
* **Facts** record *what happened*: `fact_sales_line`, `fact_purchase_line`, `fact_stock_movement`, `fact_invoice_payment`.

## How this notebook works

The SQL lives in two files so it can also be reused outside the notebook:

| File | Role |
|---|---|
| `sql/schema.sql` | Builds the 9 tables and their descriptions (the A3 deliverable) |
| `sql/validate_star.sql` | Checks row counts, totals, grain and keys |

Each step below runs one of them and shows the result, followed by a **"Reading the output"** note.

## Data preservation

Everything is created in the separate schema `star`. The raw tables in `main` are never modified, and no row is dropped: problems stay visible through flag columns such as `is_cancelled` and `invoice_id_duplicated`.

## Contents

0. Setup · 1. Check inputs · 2. Build the schema · 3. Inspect what was built · 4. Validate · 5. Read the descriptions · 6. Preview the data · 7. Summary

## 0. Setup

**Objective.** Load the libraries, set the paths and open the database.

**Important.** Run Jupyter from the **project folder**, the one that contains `warehouse.duckdb`, `sql/` and `data/processed/`. The connection here is read-write because the notebook creates the `star` schema, but it only writes inside `star`.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

DB_PATH = "warehouse.duckdb"
SQL_DIR = Path("sql")
PROCESSED = Path("data/processed")

# Close any connection left open by an earlier notebook in this kernel
try:
    con.close()
except Exception:
    pass

star_con = duckdb.connect(DB_PATH)      # read-write; only schema `star` is written
con = star_con                          # same name used in the Week 2 notebook
print("Connected to", DB_PATH)

Connected to warehouse.duckdb


**Reading the output.** `Connected to warehouse.duckdb`. If you get a *file locked* error, another notebook still has the database open: restart that kernel and run this cell again.

## 1. Check the inputs

**Objective.** Confirm the 12 Parquet files from Week 2 exist before building anything.

**Why.** The schema reads these files directly. If one is missing, the build fails halfway with a confusing message. A quick check gives a clear one instead.

In [2]:
EXPECTED = ["branches", "products", "inventory_master", "suppliers", "customers",
            "invoices", "payments", "purchase_orders_header", "purchase_orders_lines",
            "sales_orders_header", "sales_orders_lines", "stock_ledger"]

missing = [t for t in EXPECTED if not (PROCESSED / f"{t}.parquet").exists()]
for f in ["schema.sql", "validate_star.sql"]:
    if not (SQL_DIR / f).exists():
        missing.append(f"sql/{f}")

assert not missing, f"Missing: {missing}. Run `python -m src.cleaning` and check the sql/ folder."
print(f"All {len(EXPECTED)} Parquet files and both SQL files found.")

All 12 Parquet files and both SQL files found.


**Reading the output.** `All 12 Parquet files and both SQL files found.` If an `AssertionError` appears, it lists what is missing. Missing Parquet files mean Week 2's `python -m src.cleaning` has not been run; missing SQL files mean they are not in the `sql/` folder.

## 2. Build the star schema

**Objective.** Run `sql/schema.sql`, which creates the schema `star` with 5 dimension tables and 4 fact tables.

**Design rules** (all explained in comments at the top of `schema.sql`):

1. Natural keys (`product_id`, `branch_id`, ...) are used because they are unique and small.
2. Dates become integer keys `YYYYMMDD` that join to `dim_date`.
3. Cancelled orders and re-issued IDs are **kept and flagged**, never deleted.
4. Invoices are keyed on `so_id`, not `invoice_id` (197 re-issued IDs).
5. `payment_status_derived` and `lifetime_value_with_gst` replace the unreliable stored fields.

Running the cell rebuilds all nine tables from scratch (`CREATE OR REPLACE`), so it is safe to run again.

In [3]:
schema_sql = (SQL_DIR / "schema.sql").read_text(encoding="utf-8")
con.execute(schema_sql)
print("Star schema built.")

Star schema built.


**Reading the output.** `Star schema built.` and nothing else. The tables now exist inside `warehouse.duckdb`. Any red error points to a line in `schema.sql`; the most common cause is a missing Parquet column from Week 2.

## 3. Inspect what was built

**Objective.** List every table in `star` with its row count and column count.

**Expected row counts** from the Week 2 profile: `fact_sales_line` 130,402 · `fact_purchase_line` 155,495 · `fact_stock_movement` 237,230 · `fact_invoice_payment` 18,033 · `dim_product` 30 · `dim_branch` 6 · `dim_customer` 500 · `dim_supplier` 8 · `dim_date` 2,557 days (2019-01-01 to 2025-12-31).

In [4]:
tables = con.execute("""
    SELECT table_name, column_count
    FROM duckdb_tables()
    WHERE schema_name = 'star'
    ORDER BY CASE WHEN table_name LIKE 'dim_%' THEN 0 ELSE 1 END, table_name
""").df()

tables["rows"] = [con.execute(f"SELECT COUNT(*) FROM star.{t}").fetchone()[0]
                  for t in tables["table_name"]]
tables["type"] = tables["table_name"].str.split("_").str[0].map({"dim": "Dimension", "fact": "Fact"})
tables[["type", "table_name", "rows", "column_count"]]

,type,table_name,rows,column_count
0,Dimension,dim_branch,6,14
1,Dimension,dim_customer,500,18
2,Dimension,dim_date,2557,13
3,Dimension,dim_product,30,28
4,Dimension,dim_supplier,8,12
5,Fact,fact_invoice_payment,18033,17
6,Fact,fact_purchase_line,155495,16
7,Fact,fact_sales_line,130402,21
8,Fact,fact_stock_movement,237230,11


**Reading the output.** Nine rows: five **Dimensions** then four **Facts**, each with its row count and number of columns. Compare the row counts with the expected figures above. A fact table with fewer rows than its source means a join dropped records; the validation in the next step proves this formally.

## 4. Validate

**Objective.** Run `sql/validate_star.sql`. This is the sprint's definition of done: *each fact table reconciles to source row counts and totals*.

**Four groups of checks.**

| Group | Question |
|---|---|
| Row counts | Does each fact have exactly as many rows as its source? |
| Totals | Do money and quantity sums equal the source? |
| Grain | Is each fact's key really unique (0 surplus rows)? |
| Foreign keys | Does every fact key find its dimension row (0 orphans)? |

In [5]:
validate_sql = (SQL_DIR / "validate_star.sql").read_text(encoding="utf-8")
validation = con.execute(validate_sql).df()
validation

,check_name,expected,actual,status
0,rows: fact_sales_line = sales_orders_lines,1.304020e+05,1.304020e+05,PASSED
1,rows: fact_purchase_line = purchase_orders_lines,1.554950e+05,1.554950e+05,PASSED
2,rows: fact_stock_movement = stock_ledger,2.372300e+05,2.372300e+05,PASSED
3,rows: fact_invoice_payment = invoices,1.803300e+04,1.803300e+04,PASSED
4,rows: dim_product = products,3.000000e+01,3.000000e+01,PASSED
5,rows: dim_branch = branches,6.000000e+00,6.000000e+00,PASSED
6,rows: dim_customer = customers,5.000000e+02,5.000000e+02,PASSED
7,rows: dim_supplier = suppliers,8.000000e+00,8.000000e+00,PASSED
8,total: sales line_grand_total,3.242475e+10,3.242475e+10,PASSED
9,total: sales header grand_total = fact (all orders),3.242475e+10,3.242475e+10,PASSED


**Reading the output.** One row per check with `expected`, `actual` and `status`. **Every status should be `PASSED`.** A `FAILED` row names the problem: a `rows:` or `total:` failure means data was lost or duplicated in a join; a `grain:` failure means the key is not unique; an `fk:` failure means a fact row has no matching dimension row.

In [6]:
failed = validation[validation["status"] != "PASSED"]
print(f"{len(validation) - len(failed)} of {len(validation)} checks passed.")
assert failed.empty, "Some checks failed - see the table below."

36 of 36 checks passed.


**Reading the output.** `N of N checks passed.` The `assert` stops the notebook if any check failed, so you cannot carry on by accident with a broken schema. If it stops, scroll up to the validation table and look for `FAILED`.

## 5. Read the descriptions

**Objective.** Show the description stored on every table and column.

**Why.** `schema.sql` attaches a description to each table and column (`COMMENT ON`). These travel with the database and are the starting point for the **data dictionary (A6)**. First the table descriptions, then the columns of one table as an example.

In [7]:
con.execute("""
    SELECT table_name, comment AS description
    FROM duckdb_tables()
    WHERE schema_name = 'star'
    ORDER BY table_name
""").df()

,table_name,description
0,dim_branch,"Branch/warehouse dimension, one row per branch (6). Key: branch_id."
1,dim_customer,"Customer dimension, one row per customer (500). Key: customer_id."
2,dim_date,"Calendar dimension, one row per day 2019-01-01 to 2025-12-31. Key: date_key (YYYYMMDD)."
3,dim_product,"Product dimension, one row per spare part (30). Key: product_id."
4,dim_supplier,"Supplier dimension, one row per supplier (8). Key: supplier_id."
5,fact_invoice_payment,"Grain: one invoice, keyed by so_id (invoice_id is not unique). Invoice value, payments..."
6,fact_purchase_line,"Grain: one purchase order line (po_id, line_number). Procurement quantity, cost and da..."
7,fact_sales_line,"Grain: one sales order line (so_id, line_number). Revenue, GST, cost and margin per li..."
8,fact_stock_movement,Grain: one stock ledger movement (movement_id). IN / OUT / ADJUSTMENT with running bal...


In [8]:
# Change the table name to read any other table's column descriptions
con.execute("""
    SELECT column_name, data_type, comment AS description
    FROM duckdb_columns()
    WHERE schema_name = 'star' AND table_name = 'fact_invoice_payment'
    ORDER BY column_index
""").df()

,column_name,data_type,description
0,so_id,VARCHAR,Sales order ID. Unique per invoice: this is the reliable invoice key.
1,invoice_id,VARCHAR,Invoice ID. NOT unique (197 re-issued IDs); do not join on it alone.
2,customer_id,VARCHAR,FK to dim_customer.
3,branch_id,VARCHAR,FK to dim_branch.
4,invoice_date_key,BIGINT,FK to dim_date: invoice date.
5,due_date_key,BIGINT,FK to dim_date: payment due date.
6,total_order_value,BIGINT,"Invoice value before GST, INR."
7,total_gst_amount,DOUBLE,"GST on the invoice, INR."
8,grand_total,DOUBLE,"Invoice value including GST, INR."
9,payment_status_stored,VARCHAR,ORIGINAL stored status. 67 stored as Paid are only partly paid. Do not use.


**Reading the output.** The first table gives a one-line description of each of the nine tables, including its **grain** (what one row represents). The second lists every column of `fact_invoice_payment` with its type and meaning. Note the warnings on `invoice_id`, `payment_status_stored` and the derived columns: they encode the Week 2 decisions. Dimension columns that came straight from the raw data have no description yet; the data dictionary (A6) will fill these in.

## 6. Preview the data

**Objective.** Look at three rows of each fact table to confirm the columns hold sensible values.

**What to look for.** Date keys such as `20230415`, `is_cancelled` as `True/False`, `gross_margin` as a number, `signed_quantity` empty for adjustments.

In [9]:
for t in ["fact_sales_line", "fact_purchase_line", "fact_stock_movement", "fact_invoice_payment"]:
    print(f"\n{t}")
    display(con.execute(f"SELECT * FROM star.{t} LIMIT 3").df())


fact_sales_line


,so_id,line_number,customer_id,branch_id,product_id,order_date_key,delivery_date_key,order_status,is_cancelled,sales_channel,payment_terms,quantity,unit_price,gst_rate,line_total,gst_amount,line_grand_total,unit_cost_master,cogs,gross_margin,days_to_deliver
0,SO-990591,1,C0070,AHM001,P018,20220524,20220528,Delivered,False,Counter Sale,Net 30,19,11300,28,214700,60116.0,274816.0,8200,155800,58900,4
1,SO-990591,2,C0070,AHM001,P026,20220524,20220528,Delivered,False,Counter Sale,Net 30,2,310,18,620,111.6,731.6,180,360,260,4
2,SO-990591,3,C0070,AHM001,P026,20220524,20220528,Delivered,False,Counter Sale,Net 30,3,310,18,930,167.4,1097.4,180,540,390,4



fact_purchase_line


,po_id,line_number,supplier_id,branch_id,product_id,order_date_key,expected_date_key,received_date_key,po_status,is_cancelled,quantity,unit_cost,gst_rate,line_total,gst_amount,line_grand_total
0,PO-141186,1,SUP0001,PUN001,P004,20190410,20190429,20190429,Received,False,234,34709.54,28,8122032.36,2.274169e+06,1.039620e+07
1,PO-141186,2,SUP0001,PUN001,P022,20190410,20190429,20190429,Received,False,79,12317.84,28,973109.36,2.724706e+05,1.245580e+06
2,PO-141186,3,SUP0001,PUN001,P030,20190410,20190429,20190429,Received,False,46,13840.45,28,636660.70,1.782650e+05,8.149257e+05



fact_stock_movement


,movement_id,product_id,branch_id,movement_date_key,movement_type,quantity,signed_quantity,direction_known,reference_type,reference_id,running_balance
0,MOV-OUT-184742,P001,AHM001,20190103,OUT,8,-8,True,SO,SO-157767,228
1,MOV-OUT-208422,P001,AHM001,20190105,OUT,12,-12,True,SO,SO-168735,216
2,MOV-IN-111128,P001,AHM001,20190123,IN,174,174,True,PO,PO-181094,390



fact_invoice_payment


,so_id,invoice_id,customer_id,branch_id,invoice_date_key,due_date_key,total_order_value,total_gst_amount,grand_total,payment_status_stored,payment_status_derived,total_paid,payment_count,outstanding_balance,invoice_id_duplicated,first_payment_date_key,last_payment_date_key
0,SO-990591,INV-325361,C0070,AHM001,20220528,20220627,1345720,374439.6,1720159.6,Unpaid,Unpaid,0.0,0,1720159.6,False,<NA>,<NA>
1,SO-431510,INV-568476,C0031,KOL001,20220508,20220607,2194850,609423.0,2804273.0,Unpaid,Unpaid,0.0,0,2804273.0,False,<NA>,<NA>
2,SO-495859,INV-786913,C0013,AHM001,20231002,20231101,2620800,733824.0,3354624.0,Paid,Fully paid,3354624.0,1,0.0,False,20231007,20231007


**Reading the output.** Three rows from each fact table. Check that: date columns are 8-digit keys (or empty for cancelled POs' `received_date_key`), `gross_margin = line_total - cogs`, and `payment_status_derived` shows labels like `Fully paid` or `Ambiguous (duplicate invoice_id)`.

In [10]:
# Quick business sanity check: delivered revenue by fiscal year, joined through dim_date
con.execute("""
    SELECT d.fiscal_year,
           COUNT(DISTINCT f.so_id)            AS orders,
           ROUND(SUM(f.line_total) / 1e7, 2)  AS revenue_crore_before_gst,
           ROUND(SUM(f.gross_margin) / 1e7, 2) AS gross_margin_crore
    FROM star.fact_sales_line f
    JOIN star.dim_date d ON f.order_date_key = d.date_key
    WHERE NOT f.is_cancelled
    GROUP BY d.fiscal_year
    ORDER BY d.fiscal_year
""").df()

,fiscal_year,orders,revenue_crore_before_gst,gross_margin_crore
0,2019,740,95.80,20.54
1,2020,2984,380.03,82.05
2,2021,3089,400.99,86.46
3,2022,3032,382.08,82.50
4,2023,2931,380.09,81.76
5,2024,3063,383.13,82.71
6,2025,2194,273.78,58.98


**Reading the output.** Orders, revenue and gross margin per Indian fiscal year (April–March), in crore INR (1 crore = 10 million), with cancelled orders excluded. This proves the fact and the date dimension work together. Check that each year has a plausible number of orders and that margin is below revenue. These are early looks, not formal KPIs; the baseline KPIs come later this week.

## 7. Summary

| Item | Result |
|---|---|
| Schema | `star` inside `warehouse.duckdb`: 5 dimensions, 4 facts |
| Validation | All checks passed (row counts, totals, grain, foreign keys) |
| Descriptions | Every table, and every fact column, described in the database |
| Decisions carried | Cancelled rows flagged, invoices keyed on `so_id`, derived payment status, recomputed customer value |

**Status: COMPLETED** once the validation cell reports all checks passed.

**Hand-off.** Feature engineering (`src/features.py`), the data dictionary (A6) and the baseline KPIs now read from the `star` tables. Close the connection below when you are done.

In [11]:
con.close()
print("Connection closed.")

Connection closed.
